In [0]:
# Service Principal credentials
application_id = "*******************************"
authentication_key = "***************************"
tenant_id = "*******************************"

# Set Spark config for ADLS access
spark.conf.set("fs.azure.account.auth.type.delearn.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.delearn.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.delearn.dfs.core.windows.net", application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.delearn.dfs.core.windows.net", authentication_key)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.delearn.dfs.core.windows.net", "https://login.microsoftonline.com/" + tenant_id + "/oauth2/token")

print("✅ Spark config set!")

In [0]:
dbutils.widgets.text("silver_table_name", "scd_project.silver_customer_incremental")
dbutils.widgets.text("gold_table_name", "scd_project.dim_customer_scd2")
dbutils.widgets.text("batch_id", "")

silver_table_name = dbutils.widgets.get("silver_table_name")
gold_table_name = dbutils.widgets.get("gold_table_name")
batch_id = dbutils.widgets.get("batch_id")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.sql("CREATE DATABASE IF NOT EXISTS scd_project")

In [0]:
silver_batch_df = spark.table(silver_table_name) \
    .filter(col("BatchID") == batch_id)

display(silver_batch_df.orderBy("CustomerID"))

In [0]:
gold_exists = spark.catalog.tableExists(gold_table_name)

print("Gold table exists:", gold_exists)

In [0]:
if not gold_exists:
    window_spec = Window.orderBy("CustomerID")

    gold_initial_df = silver_batch_df \
        .withColumn("SurrogateKey", row_number().over(window_spec)) \
        .withColumn("EffectiveStartDate", current_timestamp()) \
        .withColumn("EffectiveEndDate", to_timestamp(lit("9999-12-31 00:00:00"))) \
        .withColumn("IsCurrent", lit(1)) \
        .withColumn("GoldCreatedDate", current_timestamp()) \
        .withColumn("GoldUpdatedDate", lit(None).cast("timestamp")) \
        .select(
            "SurrogateKey",
            "CustomerID",
            "CustomerName",
            "City",
            "Email",
            "Phone",
            "EffectiveStartDate",
            "EffectiveEndDate",
            "IsCurrent",
            "HashValue",
            "CreatedDate",
            "UpdatedDate",
            "SourceSystem",
            "BatchID",
            "IngestionDate",
            "GoldCreatedDate",
            "GoldUpdatedDate"
        )

    
    gold_initial_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(gold_table_name)


    print("Initial Gold SCD2 load completed")

In [0]:
if gold_exists:
    current_gold_df = spark.table(gold_table_name) \
        .filter(col("IsCurrent") == 1) \
        .select(
            col("CustomerID").alias("GoldCustomerID"),
            col("HashValue").alias("GoldHashValue")
        )

    compare_df = silver_batch_df.alias("S") \
        .join(
            current_gold_df.alias("G"),
            col("S.CustomerID") == col("G.GoldCustomerID"),
            "left"
        )

    changed_records_df = compare_df \
        .filter(
            col("G.GoldCustomerID").isNotNull() &
            (col("S.HashValue") != col("G.GoldHashValue"))
        ) \
        .select(col("S.CustomerID")) \
        .distinct()

    records_to_insert_df = compare_df \
        .filter(
            col("G.GoldCustomerID").isNull() |
            (col("S.HashValue") != col("G.GoldHashValue"))
        ) \
        .select(
            col("S.CustomerID"),
            col("S.CustomerName"),
            col("S.City"),
            col("S.Email"),
            col("S.Phone"),
            col("S.HashValue"),
            col("S.CreatedDate"),
            col("S.UpdatedDate"),
            col("S.SourceSystem"),
            col("S.BatchID"),
            col("S.IngestionDate")
        )

    changed_records_df = changed_records_df.cache()
    records_to_insert_df = records_to_insert_df.cache()

    print("Changed records count:", changed_records_df.count())
    print("Records to insert count:", records_to_insert_df.count())

    gold_delta_table = DeltaTable.forName(spark, gold_table_name)

    gold_delta_table.alias("T") \
        .merge(
            changed_records_df.alias("S"),
            "T.CustomerID = S.CustomerID AND T.IsCurrent = 1"
        ) \
        .whenMatchedUpdate(
            set={
                "EffectiveEndDate": "current_timestamp()",
                "IsCurrent": "0",
                "GoldUpdatedDate": "current_timestamp()"
            }
        ) \
        .execute()

    max_surrogate_key = spark.table(gold_table_name) \
        .agg(max("SurrogateKey")) \
        .collect()[0][0]

    if max_surrogate_key is None:
        max_surrogate_key = 0

    window_spec = Window.orderBy("CustomerID")

    records_to_insert_with_sk_df = records_to_insert_df \
        .withColumn("RowNumber", row_number().over(window_spec)) \
        .withColumn("SurrogateKey", col("RowNumber") + lit(max_surrogate_key)) \
        .drop("RowNumber") \
        .withColumn("EffectiveStartDate", current_timestamp()) \
        .withColumn("EffectiveEndDate", to_timestamp(lit("9999-12-31 00:00:00"))) \
        .withColumn("IsCurrent", lit(1)) \
        .withColumn("GoldCreatedDate", current_timestamp()) \
        .withColumn("GoldUpdatedDate", lit(None).cast("timestamp")) \
        .select(
            "SurrogateKey",
            "CustomerID",
            "CustomerName",
            "City",
            "Email",
            "Phone",
            "EffectiveStartDate",
            "EffectiveEndDate",
            "IsCurrent",
            "HashValue",
            "CreatedDate",
            "UpdatedDate",
            "SourceSystem",
            "BatchID",
            "IngestionDate",
            "GoldCreatedDate",
            "GoldUpdatedDate"
        )

    
    records_to_insert_with_sk_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(gold_table_name)


    print("Incremental Gold SCD2 load completed")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6707552697879691>, line 1
----> 1 if gold_exists:
      2     current_gold_df = spark.table(gold_table_name) \
      3         .filter(col("IsCurrent") == 1) \
      4         .select(
      5             col("CustomerID").alias("GoldCustomerID"),
      6             col("HashValue").alias("GoldHashValue")
      7         )
      9     compare_df = silver_batch_df.alias("S") \
     10         .join(
     11             current_gold_df.alias("G"),
     12             col("S.CustomerID") == col("G.GoldCustomerID"),
     13             "left"
     14         )

NameError: name 'gold_exists' is not defined

In [0]:

gold_reference_path = f"abfss://scdgold@delearn.dfs.core.windows.net/dim_customer_scd2/batch_id={batch_id}/"

gold_reference_df = spark.table(gold_table_name)

gold_reference_df.write \
    .mode("overwrite") \
    .parquet(gold_reference_path)


In [0]:
# Write current customer reference parquet

gold_current_reference_path = f"abfss://scdgold@delearn.dfs.core.windows.net/current_customer/batch_id={batch_id}/"

gold_current_df = spark.table(gold_table_name) \
    .filter(col("IsCurrent") == 1)

gold_current_df.write \
    .mode("overwrite") \
    .parquet(gold_current_reference_path)

print("Gold current customer parquet written to:", gold_current_reference_path)

In [0]:
from pyspark.sql.functions import col

gold_total_count = spark.table("scd_project.dim_customer_scd2").count()

gold_current_count = spark.table("scd_project.dim_customer_scd2") \
    .filter(col("IsCurrent") == 1) \
    .count()

gold_history_count = spark.table("scd_project.dim_customer_scd2") \
    .filter(col("IsCurrent") == 0) \
    .count()

print("Gold total count:", gold_total_count)
print("Gold current count:", gold_current_count)
print("Gold history count:", gold_history_count)

Gold total count: 9
Gold current count: 6
Gold history count: 3
